# ✅ Solutions — ch04_web_ontology_languages — agentic lab

This is the **solution** notebook: the same lab with all 3 tasks worked. It runs clean end to end, which is what proves the reference implementations satisfy the marking scheme.

> Student version: [`05_agentic_lab.ipynb`](05_agentic_lab.ipynb)

# Chapter 4 — The Web Ontology Languages
### Notebook 5 · Agentic lab — axiomatisation under profile constraints

*Book reference: Extends §4.2 (OWL 2 features and profiles)*

Notebooks 1–4 taught you to *read and write* OWL 2 and to reason about its profiles. This lab builds an agent that does the writing — and, crucially, one that must respect the profile it was asked to target.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch4_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course, pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Turn §4.2's profile restrictions into a **reward signal** an optimiser can act on.
2. Build a task whose score splits *faithfulness* from *profile compliance*, and see why that split matters.
3. Formalise axiomatisation as a **construction MDP** — where actions change the artefact — and contrast it with Chapter 1's evidence-gathering MDP.
4. Optimise the axiomatiser with GEPA and read which OWL 2 rules it learned.

> **Prerequisite.** This lab assumes Chapter 1 Notebook 5, which introduced tools, metrics, MDPs, GEPA and skills. Here we change the *task* and keep the discipline.

## 1. The task: requirement → axiom, inside a profile

A domain expert writes a requirement in English. The agent must emit an OWL axiom that is both a faithful reading **and** legal in the OWL 2 profile the project has committed to.

These two goals genuinely conflict, which is what makes the task worth studying. 'Giraffes eat nothing but leaves' is faithfully a *universal* restriction — and OWL 2 **EL has no universal restrictions at all**. An agent that only optimises faithfulness will hand you an ontology that falls out of your profile and loses you the polynomial-time reasoning you chose EL for.

In [ ]:
print('Axiom shapes the agent may emit:', A.AXIOM_OPERATORS)
print()
print('Which profiles admit each shape (teaching-grade simplification):')
for op, profiles in A.PROFILE_TABLE.items():
    print(f'  {op:12s} {sorted(profiles)}')
print()
print('Note: EL lacks universal restrictions; QL lacks existential restrictions')
print('in the subclass position. Those two facts drive the whole exercise.')

In [ ]:
for r in A.REQUIREMENTS[:5]:
    print(f"{r['id']:22s} [{r['profile']}]  {r['text']}")
    print(f"{'':22s} gold: {r['axiom']}")

### The axiom compiler and the reasoner are the tools

The agent emits *structure*, not OWL syntax. Deterministic code compiles that structure into real triples and a reasoner checks what it entails. Same division of labour as Chapter 1: **tools do what is mechanical, the model does what is interpretive** — which is also what keeps the output space small enough to grade.

In [ ]:
ax = A.Axiom('Giraffe', 'some', 'Leaf', 'eats')
print('axiom      :', ax)
print('profiles OK:', sorted(A.profiles_allowing(ax)))
g = A.axiom_to_graph(ax)
print('\ncompiled to OWL:')
print(g.serialize(format='turtle'))

In [ ]:
chain = [A.Axiom('Giraffe', 'subclassof', 'Herbivore'),
         A.Axiom('Herbivore', 'subclassof', 'Animal')]
print('Giraffe SubClassOf Animal entailed?',
      A.entails_subclass(chain, 'Giraffe', 'Animal'))
print('...from only the first axiom?',
      A.entails_subclass(chain[:1], 'Giraffe', 'Animal'))

## 2. The metric: faithfulness and compliance, scored separately

Half the mark for the right axiom, half for staying in profile. Collapsing these into one number would hide *which* half failed — and a metric that cannot say which half failed cannot drive GEPA (Ch. 1 §4.4).

In [ ]:
train, dev = A.build_dataset('train'), A.build_dataset('dev')
print(f'train {len(train)}, dev {len(dev)}')

class Faithful:  # right reading, wrong profile
    axiom = json.dumps({'subject': 'Giraffe', 'operator': 'only',
                        'property': 'eats', 'filler': 'Leaf'})
example = [e for e in train if e.requirement_id == 'giraffe-eats-only'][0]
el_version = example.copy(profile='EL')   # same requirement, stricter profile
for label, ex in [('target RL', example), ('target EL', el_version)]:
    r = A.axiom_scorer(ex, Faithful())
    print(f'{label}: score={r.score}  violated={r.violated}')
    for n in r.notes: print('   ', n)

The *same* axiom scores 1.0 against RL and 0.5 against EL. That is the profile trade-off from §4.2 turned into a gradient the optimiser can follow.

## 3. Baseline, then GEPA

In [ ]:
lm = llm.configure_dspy(A.AXIOM_RULEBOOK, A.axiom_responder)
baseline = A.AxiomProgram()
for ex in dev:
    pred = baseline(**ex.inputs())
    print(f'{ex.requirement_id:22s} -> {A.Axiom.parse(pred.axiom)}')
before = ev.evaluate_dataset(baseline, dev, A.axiom_scorer)
print('\nBEFORE:', before['mean_score'], before['violations'])

In [ ]:
metric = ev.make_gepa_metric(A.axiom_scorer, A.AXIOM_RULEBOOK)
reflect = llm.reflection_lm(A.AXIOM_RULEBOOK, A.axiom_responder)
tuned = opt.run_gepa(baseline, train, metric, valset=train,
                     max_metric_calls=40, reflection_lm=reflect)
result = opt.compare(A.AxiomProgram(), tuned, dev, A.axiom_scorer)
print(result.report())

Read the diff: the optimiser recovered, from failure feedback alone, four rules that Chapter 4 spends pages establishing — the existential/universal distinction, is-a versus instance-of, and the profile restrictions. It did not *understand* OWL 2; it responded to a metric that punished each error specifically. That is worth being clear-eyed about: **the knowledge came from the metric**, and the metric came from you.

## 4. A construction MDP

Chapter 1's MDP had actions that only *bought information*. Here an action **changes the artefact**, so the formalisation differs:

| | Ch. 1 (evidence) | Ch. 4 (construction) |
|---|---|---|
| **S** | evidence gathered | axioms asserted so far |
| **A** | run a tool, or submit | assert a candidate axiom, or submit |
| **T** | deterministic | deterministic |
| **R** | −cost; score on submit | −cost; **entailment coverage − profile penalty** |

The reward now contains a *penalty term*, because a wrong action here does not merely waste money — it damages the artefact.

In [ ]:
candidates = [
    A.Axiom('Giraffe', 'subclassof', 'Herbivore'),
    A.Axiom('Herbivore', 'subclassof', 'Animal'),
    A.Axiom('Giraffe', 'only', 'Leaf', 'eats'),     # illegal in EL
    A.Axiom('Giraffe', 'some', 'Leaf', 'eats'),     # legal in EL
]
M = A.AxiomConstructionMDP(candidates, required_entailments=[('Giraffe', 'Animal')],
                           profile='EL', step_cost=0.05, profile_penalty=0.5)
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'|S| = {len(M.states())}   V*(s0) = {V[s0]:.3f}')
plan = mdp.run_episode(M, mdp.greedy_policy(pi))
for a in plan.actions:
    print('  ', 'submit' if a == 'submit' else str(candidates[int(a.split(":")[1])]))

The optimal plan asserts the two subsumptions that produce the required entailment **through a reasoner**, and declines both `eats` axioms — they cost money, add no required entailment, and one of them would have cost an extra 0.5 for leaving EL. Notice that the agent is rewarded for exploiting *entailment* rather than asserting `Giraffe SubClassOf Animal` directly: that is precisely the argument of Chapter 4 §4.3, expressed as a policy.

In [ ]:
print('what happens if we drop the profile penalty:')
M2 = A.AxiomConstructionMDP(candidates, [('Giraffe', 'Animal')],
                            profile='EL', step_cost=0.05, profile_penalty=0.0)
V2, pi2 = mdp.value_iteration(M2)
print('  V* =', round(V2[M2.initial_state()], 3),
      '| plan:', mdp.run_episode(M2, mdp.greedy_policy(pi2)).actions)
print('\nThe plan is unchanged -- the illegal axiom was already not worth its\n'
      'step cost. A penalty only changes behaviour when the illegal action is\n'
      'otherwise attractive; see Exercise 4.2.')

### Task 4.1 — Try to make the profile penalty bite

Set up a case where an EL-illegal axiom is available, sweep `profile_penalty` from 0 to 1, and report the penalty at which the optimal policy stops using it. Then explain your result — it is probably not the one you expected.

> **Hint.** Build the MDP with only two candidates and sweep `profile_penalty`.

In [ ]:
cands = [A.Axiom('Giraffe', 'only', 'Leaf', 'eats'),
         A.Axiom('Giraffe', 'subclassof', 'Herbivore')]
req = [('Giraffe', 'Herbivore')]
rows = []
for penalty in [0.0, 0.25, 0.5, 1.0]:
    Mp = A.AxiomConstructionMDP(cands, req, profile='EL',
                                step_cost=0.05, profile_penalty=penalty)
    Vp, pip = mdp.value_iteration(Mp)
    ep = mdp.run_episode(Mp, mdp.greedy_policy(pip))
    asserted = [str(cands[int(a.split(':')[1])]) for a in ep.actions if a != 'submit']
    illegal = [a for a in asserted if 'only' in a]
    rows.append({'penalty': penalty, 'V*': round(Vp[Mp.initial_state()], 3),
                 'n_asserted': len(asserted), 'breaks_EL': bool(illegal)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nThe subsumption axiom alone satisfies the requirement legally, so the\n'
      'optimal policy never needs the EL-illegal axiom -- the penalty is not what\n'
      'protects the profile here; having a legal alternative is. That is the real\n'
      'lesson: reward shaping cannot rescue a candidate set with no legal option.')

**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert [r['penalty'] for r in rows] == [0.0, 0.25, 0.5, 1.0]
assert all(r['n_asserted'] >= 1 for r in rows), 'the requirement must still be met'
# The lesson: a legal alternative exists, so the penalty never has to bite.
assert not any(r['breaks_EL'] for r in rows)
# Raising a penalty cannot raise the optimal value.
assert rows[-1]['V*'] <= rows[0]['V*'] + 1e-9

### Task 4.2 — Force the conflict

Now construct a case where **no** legal axiom satisfies the requirement, and decide what the agent should do. Argue for a reward design that makes the right choice optimal.

In [ ]:
# Only an EL-illegal axiom can produce the required entailment.
cands = [A.Axiom('Giraffe', 'only', 'Leaf', 'eats')]
req = [('Giraffe', 'Leaf')]      # not entailed by a universal restriction
for penalty in [0.0, 0.5]:
    Mx = A.AxiomConstructionMDP(cands, req, profile='EL',
                                step_cost=0.05, profile_penalty=penalty)
    Vx, pix = mdp.value_iteration(Mx)
    ep = mdp.run_episode(Mx, mdp.greedy_policy(pix))
    print(f'penalty={penalty}: V*={Vx[Mx.initial_state()]:.2f} plan={ep.actions}')
print('\nAt every penalty -- including zero -- the agent submits an EMPTY ontology.\n'
      'The requirement is unsatisfiable from the available axioms, so asserting\n'
      'anything only costs. Silence is optimal here only because the reward has no\n'
      'way to express partial credit or escalation. The right engineering answer is\n'
      'to escalate (change the profile, or renegotiate the requirement). A reward\n'
      'function with only two options cannot express that, so a third action --\n'
      'ESCALATE, with a small negative reward -- belongs in the action set. Reward\n'
      'design is where you decide what your agent is allowed to do when it cannot win.')

**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
# The requirement is unsatisfiable from the candidate set, so at every penalty --
# including zero -- the optimal policy asserts nothing and submits an empty
# ontology. Silence is optimal only because the reward cannot express escalation.
for penalty in [0.0, 0.5]:
    Mx = A.AxiomConstructionMDP(cands, req, profile='EL',
                                step_cost=0.05, profile_penalty=penalty)
    Vx, pix = mdp.value_iteration(Mx)
    plan = mdp.run_episode(Mx, mdp.greedy_policy(pix)).actions
    assert [a for a in plan if a != 'submit'] == [], (
        f'asserted an axiom at penalty={penalty}; the empty ontology is optimal')
    assert plan[-1] == 'submit'

### Task 4.3 — Add a Chapter 4 rule the optimiser must discover

OWL 2 DL forbids non-simple (e.g. transitive) properties in cardinality restrictions — the rule behind Example 4.2. Add a `simple-property-only` rule and a requirement that punishes violating it, then show GEPA discovers it.

> **Hint.** Wrap `axiom_scorer`, halve the score when a transitive property appears under `only`, and add the rule id to `violated`.

In [ ]:
from oe_course.llm import Rule, RuleBook
from oe_course.evaluation import ScoreReport

TRANSITIVE = {'isPartOf'}

def strict_scorer(gold, pred):
    report = A.axiom_scorer(gold, pred)
    ax = A.Axiom.parse(getattr(pred, 'axiom', None))
    if ax and ax.property in TRANSITIVE and ax.operator == 'only':
        report.score *= 0.5
        report.notes.append(
            f"'{ax.property}' is transitive; OWL 2 DL forbids non-simple properties "
            'in universal/cardinality restrictions (Example 4.2).')
        report.violated = list(dict.fromkeys(report.violated + ['simple-property-only']))
    return report

strict_rules = RuleBook(list(A.AXIOM_RULEBOOK) + [Rule(
    'simple-property-only',
    'Never use a transitive property (such as isPartOf) inside a universal or '
    'cardinality restriction; OWL 2 DL requires simple properties there.')])

lm2 = llm.configure_dspy(strict_rules, A.axiom_responder)
reflect2 = llm.reflection_lm(strict_rules, A.axiom_responder)
strict_metric = ev.make_gepa_metric(strict_scorer, strict_rules)
tuned2 = opt.run_gepa(A.AxiomProgram(), A.build_dataset('all'), strict_metric,
                      valset=A.build_dataset('all'), max_metric_calls=50,
                      reflection_lm=reflect2)
found = strict_rules.active_in(opt.instruction_of(tuned2))
print('rules discovered:', sorted(found))
print('\nWhether simple-property-only appears depends on whether the training set\n'
      'ever punished it. Check the violation histogram before concluding the\n'
      'optimiser failed -- an undiscovered rule usually means an unrepresented case.')
print('violations seen:',
      ev.evaluate_dataset(A.AxiomProgram(), A.build_dataset('all'), strict_scorer)['violations'])

**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
from types import SimpleNamespace

assert any(r.id == 'simple-property-only' for r in strict_rules)

# The rule has to bite: a transitive property inside a universal restriction is
# exactly what OWL 2 DL forbids, so the strict scorer must penalise and record it.
# Axiom.parse takes a dict, a JSON object or an Axiom -- not the display form.
gold = A.build_dataset('all')[0]
offender = SimpleNamespace(axiom=A.Axiom('Wheel', 'only', 'Car', 'isPartOf'))
assert strict_scorer(gold, offender).score <= A.axiom_scorer(gold, offender).score
assert 'simple-property-only' in strict_scorer(gold, offender).violated

# A simple property in the same position is not penalised by the new rule.
innocent = SimpleNamespace(axiom=A.Axiom('Giraffe', 'only', 'Leaf', 'eats'))
assert 'simple-property-only' not in strict_scorer(gold, innocent).violated

## Carrying this forward

Chapters 1 and 4 now share one scaffolding and differ only in the task:

| | Ch. 1 | Ch. 4 |
|---|---|---|
| task | assess an artefact | build an axiom |
| MDP | gather evidence | construct, under constraints |
| metric | level + defect F1 | faithfulness + profile compliance |
| what GEPA learns | reporting discipline | OWL 2 semantics and profile limits |

The pattern is the deliverable. Every remaining chapter plugs a new task into it — see `course/README.md` for the task, MDP and metric proposed for each.